In [0]:
# ============================================================
# NOTEBOOK 3: ML MODELS + MLFLOW
# Team AryaBit | HackBricks 2026
# Purpose: Train LR baseline + XGBoost, log everything in MLflow
# Register best model in Unity Catalog Model Registry
# ============================================================

import logging
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from pyspark.sql.functions import col
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("AryaBit.ML")

# ── CONFIG ───────────────────────────────────────────────────
CATALOG         = "workspace"
SCHEMA          = "default"
SILVER_TABLE    = f"{CATALOG}.{SCHEMA}.silver_student_dropout"
EXPERIMENT_NAME = "/AryaBit_HackBricks_2026"
MODEL_NAME      = f"{CATALOG}.{SCHEMA}.dropout_risk_model"

# Features for training
FEATURE_COLS = [
    # Engineered features (our secret weapon)
    "grade_delta",
    "absenteeism_trend",
    "financial_stress_index",
    "overall_approval_rate",
    "avg_grade_overall",
    # Raw academic features
    "curricular_units_1st_sem_grade",
    "curricular_units_2nd_sem_grade",
    "curricular_units_1st_sem_approved",
    "curricular_units_2nd_sem_approved",
    "curricular_units_1st_sem_enrolled",
    "curricular_units_2nd_sem_enrolled",
    # Financial + demographic (used for prediction, audited for fairness)
    "admission_grade",
    "age_at_enrollment",
    "debtor",
    "tuition_fees_up_to_date",
    "scholarship_holder",
    "displaced",
]

TARGET_COL = "dropout_label"

# Sensitive cols — NOT used in training, ONLY in fairness audit
SENSITIVE_COLS = ["gender", "scholarship_holder", "debtor"]

# ── STEP 1: LOAD SILVER + PREPARE DATA ───────────────────────
def load_and_prepare_data():
    logger.info("Loading Silver table...")
    silver_df = spark.table(SILVER_TABLE)
    
    # Select only what we need
    select_cols = FEATURE_COLS + [TARGET_COL] + SENSITIVE_COLS
    # Deduplicate columns
    select_cols = list(dict.fromkeys(select_cols))
    
    pandas_df = silver_df.select(select_cols).toPandas()
    logger.info(f"Dataset shape: {pandas_df.shape}")
    
    X = pandas_df[FEATURE_COLS]
    y = pandas_df[TARGET_COL]
    sensitive = pandas_df[SENSITIVE_COLS]
    
    return X, y, sensitive, pandas_df

# ── STEP 2: TRAIN TEST SPLIT ──────────────────────────────────
def split_data(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y  # Maintain class distribution
    )
    logger.info(f"Train: {X_train.shape}, Test: {X_test.shape}")
    logger.info(f"Train dropout rate: {y_train.mean():.3f}")
    logger.info(f"Test dropout rate:  {y_test.mean():.3f}")
    return X_train, X_test, y_train, y_test

# ── STEP 3: COMPUTE METRICS ───────────────────────────────────
def compute_metrics(y_true, y_pred, y_prob):
    return {
        "accuracy":  round(accuracy_score(y_true, y_pred), 4),
        "f1_score":  round(f1_score(y_true, y_pred), 4),
        "roc_auc":   round(roc_auc_score(y_true, y_prob), 4),
        "precision": round(precision_score(y_true, y_pred), 4),
        "recall":    round(recall_score(y_true, y_pred), 4),
    }

# ── STEP 4: TRAIN LOGISTIC REGRESSION (BASELINE) ─────────────
def train_logistic_regression(X_train, X_test, y_train, y_test):
    logger.info("Training Logistic Regression baseline...")
    
    mlflow.set_experiment(EXPERIMENT_NAME)
    
    with mlflow.start_run(run_name="LogisticRegression_Baseline") as run:
        # Pipeline: scaler + model
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=1000,
                random_state=42,
                class_weight="balanced"  # Handle class imbalance
            ))
        ])
        
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1]
        
        metrics = compute_metrics(y_test, y_pred, y_prob)
        
        # Log params
        mlflow.log_params({
            "model_type": "LogisticRegression",
            "max_iter": 1000,
            "class_weight": "balanced",
            "test_size": 0.2,
            "random_state": 42,
            "n_features": len(FEATURE_COLS)
        })
        
        # Log metrics
        mlflow.log_metrics(metrics)
        
        # Log model
        mlflow.sklearn.log_model(
            pipeline,
            "logistic_regression_model",
            input_example=X_test.iloc[:3]
        )
        
        lr_run_id = run.info.run_id
        logger.info(f"LR Run ID: {lr_run_id}")
        logger.info(f"LR Metrics: {metrics}")
        
        return pipeline, metrics, lr_run_id, y_pred, y_prob

# ── STEP 5: TRAIN RANDOM FOREST ───────────────────────────────
def train_random_forest(X_train, X_test, y_train, y_test):
    logger.info("Training Random Forest...")
    
    mlflow.set_experiment(EXPERIMENT_NAME)
    
    with mlflow.start_run(run_name="RandomForest_AryaBit") as run:
        pipeline = Pipeline([
            ("model", RandomForestClassifier(
                n_estimators=200,
                max_depth=10,
                min_samples_split=5,
                random_state=42,
                class_weight="balanced",
                n_jobs=-1
            ))
        ])
        
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1]
        
        metrics = compute_metrics(y_test, y_pred, y_prob)
        
        # Feature importance
        importances = pipeline.named_steps["model"].feature_importances_
        feature_importance = dict(zip(FEATURE_COLS, importances))
        top_features = sorted(
            feature_importance.items(),
            key=lambda x: x[1],
            reverse=True
        )[:5]
        
        # Log params
        mlflow.log_params({
            "model_type": "RandomForest",
            "n_estimators": 200,
            "max_depth": 10,
            "class_weight": "balanced",
            "test_size": 0.2,
            "random_state": 42,
            "n_features": len(FEATURE_COLS)
        })
        
        # Log metrics
        mlflow.log_metrics(metrics)
        
        # Log feature importances as params
        for feat, imp in top_features:
            mlflow.log_metric(f"importance_{feat}", round(imp, 4))
        
        # Log model
        mlflow.sklearn.log_model(
            pipeline,
            "random_forest_model",
            input_example=X_test.iloc[:3]
        )
        
        rf_run_id = run.info.run_id
        logger.info(f"RF Run ID: {rf_run_id}")
        logger.info(f"RF Metrics: {metrics}")
        logger.info(f"Top 5 Features: {top_features}")
        
        return pipeline, metrics, rf_run_id, y_pred, y_prob, top_features

# ── STEP 6: COMPARE + REGISTER BEST MODEL ─────────────────────
def register_best_model(
    lr_metrics, lr_run_id,
    rf_metrics, rf_run_id
):
    logger.info("Comparing models...")
    
    print("\n" + "="*60)
    print("MODEL COMPARISON")
    print("="*60)
    print(f"{'Metric':<15} {'Logistic Reg':>15} {'Random Forest':>15}")
    print("-"*45)
    for metric in ["accuracy", "f1_score", "roc_auc", "precision", "recall"]:
        print(f"{metric:<15} {lr_metrics[metric]:>15} {rf_metrics[metric]:>15}")
    
    # Pick winner by ROC-AUC
    if rf_metrics["roc_auc"] >= lr_metrics["roc_auc"]:
        best_run_id = rf_run_id
        best_model_name = "RandomForest"
        best_metrics = rf_metrics
        logger.info("Winner: Random Forest ✅")
    else:
        best_run_id = lr_run_id
        best_model_name = "LogisticRegression"
        best_metrics = lr_metrics
        logger.info("Winner: Logistic Regression ✅")
    
    print(f"\n🏆 WINNER: {best_model_name} (ROC-AUC: {best_metrics['roc_auc']})")
    
    # Register in Unity Catalog Model Registry
    logger.info(f"Registering {best_model_name} to Unity Catalog...")
    
    model_uri = f"runs:/{best_run_id}/{best_model_name.lower().replace(' ', '_')}_model"
    
    try:
        registered = mlflow.register_model(
            model_uri=model_uri,
            name=MODEL_NAME
        )
        logger.info(f"Model registered: {MODEL_NAME} v{registered.version}")
        print(f"\n✅ Model registered in Unity Catalog: {MODEL_NAME}")
        print(f"   Version: {registered.version}")
    except Exception as e:
        logger.warning(f"Registration note: {e}")
    
    return best_run_id, best_model_name, best_metrics

# ── MAIN EXECUTION ────────────────────────────────────────────
def run_ml_pipeline():
    logger.info("Starting ML Pipeline — AryaBit")
    
    try:
        # Set MLflow to use Unity Catalog
        mlflow.set_registry_uri("databricks-uc")
        
        # Load data
        X, y, sensitive, full_df = load_and_prepare_data()
        
        # Split
        X_train, X_test, y_train, y_test = split_data(X, y)
        
        # Train LR
        lr_model, lr_metrics, lr_run_id, lr_pred, lr_prob = \
            train_logistic_regression(X_train, X_test, y_train, y_test)
        
        # Train RF
        rf_model, rf_metrics, rf_run_id, rf_pred, rf_prob, top_features = \
            train_random_forest(X_train, X_test, y_train, y_test)
        
        # Compare + Register
        best_run_id, best_model_name, best_metrics = register_best_model(
            lr_metrics, lr_run_id,
            rf_metrics, rf_run_id
        )
        
        print("\n" + "="*60)
        print("TOP 5 MOST IMPORTANT FEATURES")
        print("="*60)
        for i, (feat, imp) in enumerate(top_features, 1):
            print(f"{i}. {feat}: {round(imp*100, 2)}%")
        
        logger.info("ML Pipeline COMPLETE ✅")
        
        # Store for next notebook
        return rf_model, X_test, y_test, rf_pred, rf_prob, full_df
        
    except Exception as e:
        logger.error(f"ML Pipeline FAILED: {str(e)}")
        raise e

# ── RUN ───────────────────────────────────────────────────────
rf_model, X_test, y_test, rf_pred, rf_prob, full_df = run_ml_pipeline()